# DQN Traffic Light Control - Quick Start

This notebook demonstrates the DQN agent for traffic light optimization.

In [1]:
# Install dependencies (if needed)
# !pip install -r requirements.txt

## 1. Generate SUMO Scenario

In [2]:
from src.utils.generate_scenario import generate_simple_intersection

# Generate Vietnamese intersection with 60% motorcycles
generate_simple_intersection("data/scenarios/hn_sample")

✓ Created scenario files in data/scenarios/hn_sample/
  Next: run 'netconvert --node-files=data/scenarios/hn_sample/nodes.nod.xml --edge-files=data/scenarios/hn_sample/edges.edg.xml --output-file=data/scenarios/hn_sample/intersection.net.xml'


## 2. Build SUMO Network

Run this in terminal:
```bash
netconvert --node-files=data/scenarios/hn_sample/nodes.nod.xml \
           --edge-files=data/scenarios/hn_sample/edges.edg.xml \
           --output-file=data/scenarios/hn_sample/intersection.net.xml
```

## 3. Test Environment

In [3]:
import os
os.environ["SUMO_HOME"] = "/opt/homebrew/opt/sumo/share/sumo"  # Adjust path

from src.env.sumo_env import SumoMDPEnv, EnvConfig, VNWeights
import numpy as np

cfg = EnvConfig(
    sumocfg_path="data/scenarios/hn_sample/config.sumocfg",
    tls_id="c",
    phases=[0, 1, 2, 3],  # int phases
    action_duration=5,
    max_steps=200,
    gui=False,
    vn_weights=VNWeights(motorcycle=1.5, car=1.0, bus=2.0, truck=2.0),
)

env = SumoMDPEnv(cfg)
state = env.reset()
print(f"State dim: {env.state_dim}, Action dim: {env.action_dim}")

# Run random agent
total_r = 0
for _ in range(20):
    a = np.random.randint(env.action_dim)
    s2, r, done, info = env.step(a)
    total_r += r
    if done:
        break

env.close()
print(f"Total reward: {total_r:.2f}")

 Retrying in 1 seconds


State dim: 16, Action dim: 4
Total reward: -136.98


## 4. Train DQN Agent

In [4]:
from src.dqn.agent import DQNAgent, AgentConfig
from src.dqn.replay_buffer import ReplayBuffer
from src.utils import LinearEpsilon
import torch

# Create agent
agent_cfg = AgentConfig(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    gamma=0.99,
    lr=5e-4,
    batch_size=64,
)
agent = DQNAgent(agent_cfg)
buffer = ReplayBuffer(50000, (env.state_dim,))
eps_sched = LinearEpsilon(start=1.0, end=0.05, steps=10000)

print("Agent ready. Use train.py for full training.")

Step #100.00 (1ms ~= 1000.00*RT, ~43000.00UPS, TraCI: 2ms, vehicles TOT 102 ACT 43 BUF 7)  
Agent ready. Use train.py for full training.


## 5. Visualize Training (after running train.py)

```python
# Load trained model
agent.q.load_state_dict(torch.load("outputs/dqn_vn_tls.pt"))
agent.q.eval()

# Test with GUI
cfg.gui = True
env = SumoMDPEnv(cfg)
s = env.reset()
for _ in range(100):
    a = agent.act(s, eps=0.0)  # greedy
    s, r, d, _ = env.step(a)
    if d:
        break
env.close()
```